# 04 · Train probe
Logistic-regression probe on the saved hidden states: predict `correctness_label` (1 = hallucination) from the model's internal representation alone.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
%load_ext autoreload
%autoreload 2
from src import config

In [2]:
import json, joblib, numpy as np
from src.io_utils import read_jsonl
from src.probe import load_splits, train_probe

records = read_jsonl(config.PROBE_DATASET_PATH)
splits = load_splits(records)
{s: (X.shape, y.mean().round(3)) for s, (X, y) in splits.items()}

{'train': ((350, 2048), np.float64(0.469)),
 'val': ((75, 2048), np.float64(0.48)),
 'test': ((75, 2048), np.float64(0.427))}

In [3]:
# Pick regularisation strength on val
probe, best_C, split_metrics = train_probe(splits, verbose=True)

C=0.001: val AUROC=0.611
C=0.01: val AUROC=0.547
C=0.1: val AUROC=0.504
C=1.0: val AUROC=0.484


/Users/snehvora/Desktop/Research/Detecting LLM Hallucinations from Internal Representations Without an External Fact Checker/hallucination-research/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/snehvora/Desktop/Research/Detecting LLM Hallucinations from Internal Representations Without an External Fact Checker/hallucination-research/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/snehvora/Desktop/Research/Detecting LLM Hallucinations from Internal Representations Without an External Fact Checker/hallucination-research/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/snehvora/Desktop/Research/Detecting LLM 

In [4]:
metrics = {'model_name': config.MODEL_NAME, 'layer_index': config.LAYER_INDEX,
           'token_position_method': config.TOKEN_POSITION_METHOD, 'C': best_C, **split_metrics}

tag = f"{config.MODEL_NAME.split('/')[-1]}_L{config.LAYER_INDEX}_{config.TOKEN_POSITION_METHOD}"
config.MODELS_DIR.mkdir(exist_ok=True); config.RESULTS_DIR.mkdir(exist_ok=True)
joblib.dump(probe, config.MODELS_DIR / f'probe_{tag}.joblib')
(config.RESULTS_DIR / f'metrics_{tag}.json').write_text(json.dumps(metrics, indent=2))
metrics

/Users/snehvora/Desktop/Research/Detecting LLM Hallucinations from Internal Representations Without an External Fact Checker/hallucination-research/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/snehvora/Desktop/Research/Detecting LLM Hallucinations from Internal Representations Without an External Fact Checker/hallucination-research/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/snehvora/Desktop/Research/Detecting LLM Hallucinations from Internal Representations Without an External Fact Checker/hallucination-research/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/snehvora/Desktop/Research/Detecting LLM Hallucinations from Internal Representations Without an External Fact Checker/hallucination-research/.venv/lib/python3.9/si

{'model_name': 'Qwen/Qwen2.5-3B-Instruct',
 'layer_index': 16,
 'token_position_method': 'answer_last_token',
 'C': 0.001,
 'train': {'auroc': np.float64(0.8888014686598479),
  'accuracy': 0.7971428571428572,
  'f1': 0.778816199376947},
 'val': {'auroc': np.float64(0.611111111111111),
  'accuracy': 0.5733333333333334,
  'f1': 0.5151515151515151},
 'test': {'auroc': np.float64(0.653343023255814),
  'accuracy': 0.64,
  'f1': 0.5970149253731343}}